# Using OpenDP Synth - test

## Step 1: Install the library

It can be installed via the pip command:

In [ ]:
from rich.jupyter import print

%load_ext rich

In [ ]:
from lomas_client import Client
import numpy as np
import opendp.prelude as dp
import polars as pl
import itertools as it

## Step 2: Initialise the client

Once the library is installed, a Client object must be created. It is responsible for sending sending requests to the server and processing responses in the local environment. It enables a seamless interaction with the server. 

The client needs a few parameters to be created. Usually, these would be set in the environment by the system administrator and be transparent to lomas users. In this instance, the following code snippet sets a few of these parameters that are specific to this notebook. 

In [ ]:
# The following would usually be set in the environment by a system administrator
# and be tranparent to lomas users.
# Uncomment them if you are running against a Kubernetes deployment.
# They have already been set for you if you are running locally within a devenv or the Jupyter lab set up by Docker compose.

import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_OIDC_DISCOVERY_URL"] = "https://dex.example.com:443/.well-known/openid-configuration"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

os.environ["LOMAS_CLIENT_USER_NAME"] = "mr.corona@example.com"
os.environ["LOMAS_CLIENT_USER_PASSWORD"] = "mr.corona"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "COVID_SYNTHETIC"

# Note that all client settings can also be passed as keyword arguments to the Client constructor.
# eg. client = Client(user_name = "Mr.corona") takes precedence over setting the "LOMAS_CLIENT_USER_NAME"
# environment variable.

In [ ]:
client = Client()

## Step 3: Metadata and dummy dataset

### Getting dataset metadata

The user has never seen the data and as a first step to understand what is available to her, she would like to check the metadata of the dataset. Therefore, she just needs to call the `get_dataset_metadata()` function of the client. As this is public information, this does not cost any budget.

This function returns metadata information in a format based on [SmartnoiseSQL dictionary format](https://docs.smartnoise.org/sql/metadata.html#dictionary-format), where among other, there is information about all the available columns, their type, bound values (see Smartnoise page for more details). Any metadata is required for Smartnoise-SQL is also required here and additional information such that the different categories in a string type column column can be added.

In [ ]:
covid_metadata = client.metadata
covid_metadata


{
    '@context': ['http://www.w3.org/ns/csvw', '/home/onyxia/work/csvw-eo/csvw-eo-context.jsonld'],
    '@type': 'Table',
    'privacyUnit': 'id',
    'maxContributions': 52,
    'maxLength': 50048,
    'publicLength': 50048,
    'tableSchema': {
        'columns': [
            {
                '@type': 'Column',
                'name': 'id',
                'datatype': <DataTypes.POSITIVE_INTEGER: 'positiveInteger'>,
                'required': True,
                'privacyId': True,
                'nullableProportion': 0.0,
                'minimum': 1,
                'maximum': 2000
            },
            {
                '@type': 'Column',
                'name': 'date',
                'datatype': <DataTypes.DATE: 'date'>,
                'required': True,
                'privacyId': False,
                'nullableProportion': 0.0,
                'minimum': '2022-08-01',
                'maximum': '2023-07-30'
            },
            {
                '@type': 'C

### Get a dummy dataset

Now, that she has seen and understood the metadata, she wants to get an even better understanding of the dataset (but is still not able to see it). A solution to have an idea of what the dataset looks like it to create a dummy dataset. 

Based on the public metadata of the dataset, a random dataframe can be created created. By default, there will be 100 rows and the seed is set to 42 to ensure reproducibility, but these 2 variables can be changed to obtain different dummy datasets.
Getting a dummy dataset does not affect the budget as there is no differential privacy here. It is not a synthetic dataset and all that could be learn here is already present in the public metadata (it is created randomly on the fly based on the metadata).

Dr. FSO first create a dummy dataset with 200 rows and chooses a seed of 0.

In [ ]:
client.get_dummy_dataset()

,id,date,temporal,georegion,agegroup,sex,testType,testResult,country,subType,hospitalization,death,patient_id
0,1918,2023-01-26,29,VD,other,female,rapid_antigen_test,other,other,BQ.1,True,True,3658
1,1609,2022-11-29,50,AG,50 - 59,female,rapid_antigen_test,unknown,FL,BA.4,True,True,28084
2,1906,2022-11-16,36,TI,80+,male,pcr,positive,CH,BA.2.75,False,True,19558
3,657,2022-08-31,8,SZ,unknown,female,rapid_antigen_test,other,unknown,BA.2,True,True,10926
4,1541,2023-03-17,37,NW,0 - 9,female,rapid_antigen_test,other,unknown,BA.2,False,False,23857
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,354,2023-05-16,41,TG,unknown,unknown,pcr,positive,unknown,BA.2,False,False,5280
96,1229,2022-08-09,11,VD,20 - 29,female,pcr,positive,CH,<NA>,True,False,29599
97,1373,2023-04-26,28,AG,70 - 79,unknown,pcr,negative,other,<NA>,False,True,27604
98,1863,2023-07-14,44,TI,other,other,pcr,negative,other,<NA>,True,True,21678


In [ ]:
# MST 

columns = ['date', 'temporal', 'georegion', 'agegroup', 'sex', 'testType', 'testResult', 'country', 'subType', 'hospitalization', 'death','patient_id']
resp = client.opendp.synth(columns, epsilon=40)
print(resp)
dfs = resp.result.value

dropping {'date'} columns, no keys/cuts automatically generated nor specified


QueryResponse(
    epsilon=40.0,
    delta=0.0,
    response_type=<QueryResponseTypes.QUERY: 'query'>,
    requested_by='Mr.Corona',
    result=OpenDPPolarsQueryResult(
        type=<DPLibraries.OPENDP_POLARS: 'opendp_polars'>,
        value=shape: (50_036, 11)
┌────────────┬───────┬──────────┬─────────┬───┬─────────────┬───────────┬─────────────┬────────────┐
│ testResult ┆ death ┆ temporal ┆ country ┆ … ┆ hospitaliza ┆ georegion ┆ testType    ┆ patient_id │
│ ---        ┆ ---   ┆ ---      ┆ ---     ┆   ┆ tion        ┆ ---       ┆ ---         ┆ ---        │
│ str        ┆ bool  ┆ i64      ┆ str     ┆   ┆ ---         ┆ str       ┆ str         ┆ i64        │
│            ┆       ┆          ┆         ┆   ┆ bool        ┆           ┆             ┆            │
╞════════════╪═══════╪══════════╪═════════╪═══╪═════════════╪═══════════╪═════════════╪════════════╡
│ negative   ┆ false ┆ 1        ┆ CH      ┆ … ┆ false       ┆ GE        ┆ pcr         ┆ 1          │
│ negative   ┆ false ┆ 1        ┆ CH      ┆ … ┆ false       ┆ SZ        ┆ pcr         ┆ 1          │
│ negative   ┆ false ┆ 1        ┆ CH      ┆ … ┆ false       ┆ GL        ┆ pcr         ┆ 1327       │
│ negative   ┆ false ┆ 1        ┆ CH      ┆ … ┆ false       ┆ SH        ┆ pcr         ┆ 207        │
│ other      ┆ false ┆ 1        ┆ CH      ┆ … ┆ false       ┆ ZH        ┆ pcr         ┆ 1          │
│ …          ┆ …     ┆ …        ┆ …       ┆ … ┆ …           ┆ …         ┆ …           ┆ …          │
│ unknown    ┆ false ┆ 53       ┆ unknown ┆ … ┆ null        ┆ ZG        ┆ rapid_antig ┆ 27602      │
│            ┆       ┆          ┆         ┆   ┆             ┆           ┆ en_test     ┆            │
│ unknown    ┆ false ┆ 53       ┆ unknown ┆ … ┆ null        ┆ SG        ┆ rapid_antig ┆ 28867      │
│            ┆       ┆          ┆         ┆   ┆             ┆           ┆ en_test     ┆            │
│ unknown    ┆ false ┆ 53       ┆ unknown ┆ … ┆ true        ┆ SZ        ┆ null        ┆ 29479      │
│ unknown    ┆ true  ┆ 53       ┆ unknown ┆ … ┆ null        ┆ BE        ┆ null        ┆ 29677      │
│ null       ┆ false ┆ 53       ┆ unknown ┆ … ┆ null        ┆ ZH        ┆ null        ┆ 22459      │
└────────────┴───────┴──────────┴─────────┴───┴─────────────┴───────────┴─────────────┴────────────┘
    )
)

In [ ]:
keys, cuts = client.opendp._gen_key_cuts()
del cuts["id"]
print(keys.keys())
print(cuts)

KSsel = [pl.col(col) for col in keys.keys()] + [pl.col(col).cut(cut) for col,cut in cuts.items()]

dfs.group_by(KSsel).len()

dict_keys(['georegion', 'agegroup', 'sex', 'testType', 'testResult', 'country', 'subType', 'hospitalization', 
'death'])

{
    'temporal': [1, 6, 11, 16, 21, 26, 31, 36, 41, 46, 52],
    'patient_id': [1, 3000, 6000, 9000, 12000, 15000, 18000, 21000, 24000, 27000, 30000]
}

georegion,agegroup,sex,testType,testResult,country,subType,hospitalization,death,temporal,patient_id,len
str,str,str,str,str,str,str,bool,bool,cat,cat,u32
"""JU""","""50 - 59""",null,"""rapid_antigen_test""","""unknown""","""unknown""",null,true,false,"""(52, inf]""","""(27000, 30000]""",1
"""BS""","""20 - 29""","""male""","""pcr""","""positive""","""CH""","""BA.5""",false,false,"""(31, 36]""","""(12000, 15000]""",3
"""SG""","""80+""","""male""","""pcr""","""negative""","""CH""",null,false,false,"""(21, 26]""","""(18000, 21000]""",1
"""GR""","""30 - 39""","""male""","""pcr""","""negative""","""CH""",null,false,false,"""(41, 46]""","""(21000, 24000]""",13
"""JU""","""20 - 29""","""female""","""pcr""","""positive""","""CH""","""unknown""",false,false,"""(6, 11]""","""(24000, 27000]""",3
…,…,…,…,…,…,…,…,…,…,…,…
"""VD""","""40 - 49""","""male""","""pcr""","""positive""","""CH""","""unknown""",false,false,"""(31, 36]""","""(24000, 27000]""",4
"""UR""","""20 - 29""","""female""","""pcr""","""positive""","""CH""","""unknown""",false,false,"""(1, 6]""","""(21000, 24000]""",2
"""AG""","""10 - 19""","""female""","""pcr""","""positive""","""CH""","""BA.5""",false,false,"""(1, 6]""","""(9000, 12000]""",17


In [ ]:
from functools import reduce

excluded = ["georegion","agegroup","sex"]

kf = [pl.LazyFrame(k,schema=[col]) for col,k in keys.items() if col not in excluded] + [
    pl.LazyFrame(pl.Series(cut).cut(cut),schema=[col]) for col,cut in cuts.items() if col not in excluded]
KSselEx = [pl.col(col) for col in keys.keys() if col not in excluded] + [pl.col(col).cut(cut) for col,cut in cuts.items() if col not in excluded]


full_product = reduce(lambda acc, e: acc.join(e, how="cross"), kf)
full_product.collect()

testType,testResult,country,subType,hospitalization,death,temporal,patient_id
str,str,str,str,bool,bool,cat,cat
"""pcr""","""negative""","""CH""","""BA.1""",false,false,"""(-inf, 1]""","""(-inf, 1]"""
"""pcr""","""negative""","""CH""","""BA.1""",false,false,"""(-inf, 1]""","""(1, 3000]"""
"""pcr""","""negative""","""CH""","""BA.1""",false,false,"""(-inf, 1]""","""(3000, 6000]"""
"""pcr""","""negative""","""CH""","""BA.1""",false,false,"""(-inf, 1]""","""(6000, 9000]"""
"""pcr""","""negative""","""CH""","""BA.1""",false,false,"""(-inf, 1]""","""(9000, 12000]"""
…,…,…,…,…,…,…,…
"""rapid_antigen_test""","""unknown""","""unknown""","""unknown""",true,true,"""(46, 52]""","""(15000, 18000]"""
"""rapid_antigen_test""","""unknown""","""unknown""","""unknown""",true,true,"""(46, 52]""","""(18000, 21000]"""
"""rapid_antigen_test""","""unknown""","""unknown""","""unknown""",true,true,"""(46, 52]""","""(21000, 24000]"""


In [ ]:
plan = client.get_context(epsilon=3.0).query().group_by(KSselEx).agg([dp.len()]).with_keys(full_product)
plan.summarize()

column,aggregate,distribution,scale
str,str,str,f64
"""len""","""Frame Length""","""Integer Laplace""",17.333333


In [ ]:
dff = client.opendp.query(plan, epsilon=3.0, delta=0.0).result.value
dff

testType,testResult,country,subType,hospitalization,death,temporal,patient_id,len
str,str,str,str,bool,bool,str,str,i64
"""rapid_antigen_test""","""unknown""","""other""","""BA.1""",true,true,"""(1, 6]""","""(12000, 15000]""",28
"""pcr""","""other""","""unknown""","""unknown""",false,true,"""(-inf, 1]""","""(24000, 27000]""",13
"""pcr""","""unknown""","""other""","""other""",false,false,"""(21, 26]""","""(21000, 24000]""",0
"""rapid_antigen_test""","""positive""","""unknown""","""BA.4""",true,true,"""(-inf, 1]""","""(24000, 27000]""",0
"""rapid_antigen_test""","""other""","""unknown""","""BA.1""",false,false,"""(16, 21]""","""(9000, 12000]""",0
…,…,…,…,…,…,…,…,…
"""pcr""","""unknown""","""FL""","""other""",true,true,"""(46, 52]""","""(27000, 30000]""",0
"""pcr""","""negative""","""other""","""BA.1""",false,true,"""(46, 52]""","""(9000, 12000]""",0
"""pcr""","""negative""","""other""","""BA.2""",false,true,"""(31, 36]""","""(-inf, 1]""",0


In [ ]:
dfl = pl.scan_csv(f"{os.getenv("LOMAS_SERVER_data_directory")}/datasets/covid_synthetic_data.csv")
df = dfl.select(columns).collect()

In [ ]:
# columns = ['georegion','testType', 'testResult','country','subType','hospitalization', 'death']
# keys = pl.LazyFrame(it.product(*(client.get_column_metadata(c).get('keyValues') for c in columns)), schema=columns).collect()

# plan = client.get_context(epsilon=3.0).query().group_by(columns).agg([dp.len()]).with_keys(keys)
# plan.summarize()

In [ ]:
# dff = client.opendp.query(plan, epsilon=3.0).result.value
# dff

In [ ]:
import altair as alt
alt.data_transformers.disable_max_rows()

columns = list(set(columns) - {"date"})# - set(excluded))

#.cast({"temporal": pl.Categorical, "patient_id": pl.Categorical})
data = dfs.group_by(KSsel).len().join(df.group_by(KSsel).len(name="len_true"), on=columns, how="inner")

data_in = data.to_pandas().reset_index()
alt.Chart(data_in).mark_bar(clip=True).encode(
    x='diff:Q',
    y="count()",
    color="agegroup",
    # color=alt.Color("diff:Q").bin(maxbins=20).scale(scheme='pinkyellowgreen'),
    tooltip=(columns + ['len', 'len_true'])
).transform_calculate(diff="datum.len - datum.len_true").properties(width=1000, height=500)

alt.Chart(...)